[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-03-mcp-on-the-wire.ipynb)

# Advanced Discussion 3 — an MCP server in forty lines

A hand-written JSON-RPC MCP server over stdio, the full message transcript, and a compliance check with the official Python SDK client.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Launches a small subprocess; runs in seconds.

In [ ]:
%pip install -q mcp

In [ ]:
import json, subprocess, sys, textwrap, tempfile, os, asyncio

## 1. a complete MCP server with no SDK: newline-delimited JSON-RPC 2.0 over stdin/stdout

In [ ]:
SERVER = textwrap.dedent('''
    import sys, json
    TOOLS = [{"name": "drive_time", "description": "Driving time in minutes between two named sites.",
              "inputSchema": {"type": "object", "properties": {"origin": {"type": "string"}, "destination": {"type": "string"}}, "required": ["origin", "destination"]}}]
    TIMES = {("depot", "port"): 42, ("port", "depot"): 44}
    def reply(id_, result=None, error=None):
        msg = {"jsonrpc": "2.0", "id": id_}
        msg.update({"error": error} if error else {"result": result})
        sys.stdout.write(json.dumps(msg) + "\\n"); sys.stdout.flush()
    for line in sys.stdin:
        req = json.loads(line); method = req.get("method"); id_ = req.get("id")
        if method == "initialize":
            reply(id_, {"protocolVersion": req["params"]["protocolVersion"], "capabilities": {"tools": {}}, "serverInfo": {"name": "kestrel-demo", "version": "0.1"}})
        elif method == "notifications/initialized":
            pass                                                     # notifications have no id and get no reply
        elif method == "tools/list":
            reply(id_, {"tools": TOOLS})
        elif method == "tools/call":
            a = req["params"]["arguments"]; key = (a.get("origin", "").lower(), a.get("destination", "").lower())
            if key in TIMES: reply(id_, {"content": [{"type": "text", "text": f"{TIMES[key]} minutes"}], "isError": False})
            else: reply(id_, {"content": [{"type": "text", "text": f"no route data for {key}"}], "isError": True})
        elif id_ is not None:
            reply(id_, error={"code": -32601, "message": f"method not found: {method}"})
''')
path = os.path.join(tempfile.mkdtemp(), "kestrel_server.py"); open(path, "w").write(SERVER)

proc = subprocess.Popen([sys.executable, path], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True, bufsize=1)
def send(msg, expect_reply=True):
    proc.stdin.write(json.dumps(msg) + "\n"); proc.stdin.flush()
    print("CLIENT ->", json.dumps(msg))
    if expect_reply:
        line = proc.stdout.readline(); print("SERVER ->", line.strip()); return json.loads(line)
send({"jsonrpc": "2.0", "id": 1, "method": "initialize", "params": {"protocolVersion": "2024-11-05", "capabilities": {}, "clientInfo": {"name": "demo-client", "version": "0.1"}}})
send({"jsonrpc": "2.0", "method": "notifications/initialized"}, expect_reply=False)
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
send({"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "drive_time", "arguments": {"origin": "depot", "destination": "port"}}})
send({"jsonrpc": "2.0", "id": 4, "method": "tools/call", "params": {"name": "drive_time", "arguments": {"origin": "depot", "destination": "moon"}}})
send({"jsonrpc": "2.0", "id": 5, "method": "resources/list"})
proc.stdin.close(); proc.wait(timeout=5)

## 2. does the OFFICIAL SDK client speak to it? (a compliance check, not a dependency of the course)

In [ ]:
async def official_client():
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client
    async with stdio_client(StdioServerParameters(command=sys.executable, args=[path])) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            res = await session.call_tool("drive_time", {"origin": "port", "destination": "depot"})
            return [t.name for t in tools.tools], res.content[0].text
try: print("\nofficial SDK client ->", asyncio.run(official_client()))
except Exception as e: print("\nofficial SDK client failed:", repr(e)[:300])

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.